<a href="https://colab.research.google.com/github/hbistlin/ie332-fall2026/blob/main/Copy_of_Lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IE 332 — Lab 2: SQL Fundamentals on Boilermaker Brews

**Name:**  
**Lab section:**  

**Goal.** Query a real operational database with `SELECT`, `WHERE`, `ORDER BY`, and `LIMIT` — the clauses from SQL Lecture 2.

**How this lab works**
1. Run Section 0 to connect to the database, then work top to bottom.
2. Each exercise has a **self-check** cell.
3. When you are done, run every cell so the outputs show, save and upload to Gradescope by **Sunday 11:59pm**. Credit is for submitting.


## Section 0 — Setup

**Before anything:** upload `boilermaker_brews.db` to this Colab session (folder icon in the left sidebar → upload), or copy it in from your Drive. The file is on Brightspace.

*(Remember from lecture: the session's disk is wiped when the runtime ends: the upload is needed in every session.)*

In [2]:
# Setup: no package installation needed. This is the same %%sql command the
# lecture demo notebooks use, plus the ability to capture a result for the self-checks.
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import display

DB_PATH = Path("boilermaker_brews.db")

if not DB_PATH.exists():
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError(
            "Place boilermaker_brews.db in the notebook's working folder."
        ) from exc
    print("Choose boilermaker_brews.db from the course files.")
    files.upload()

if not DB_PATH.exists():
    raise FileNotFoundError("boilermaker_brews.db was not uploaded.")

conn = sqlite3.connect(DB_PATH)


class SQLResult(list):
    """The rows of a query, plus .keys so the self-checks can name columns."""
    def __init__(self, frame):
        super().__init__(frame.itertuples(index=False, name=None))
        self.keys = list(frame.columns)
        self.frame = frame


def _sql_magic(line, cell):
    """%%sql             run and display
       %%sql q1 <<       run, display, and store the result in q1
       %%sql q1          same thing; the << is optional"""
    target = line.replace("<<", " ").strip().split(None, 1)[0] if line.strip() else None
    statement = cell.strip()
    if statement.split(None, 1)[0].upper() in {"SELECT", "WITH", "PRAGMA", "EXPLAIN"}:
        frame = pd.read_sql_query(statement, conn)
        display(frame)
        if target:
            get_ipython().user_ns[target] = SQLResult(frame)
        return
    conn.executescript(statement)
    conn.commit()
    print("Statement executed successfully.")


get_ipython().register_magic_function(_sql_magic, "cell", "sql")

def _sql_line_magic(line):
    """%sql SELECT ...   one-liner form, used by the smoke test."""
    _sql_magic("", line)


get_ipython().register_magic_function(_sql_line_magic, "line", "sql")

n = pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM sqlite_master WHERE type='table';", conn
).iloc[0, 0]
print(f"Connected to {DB_PATH.name}: {n} tables. The %%sql command is ready.")


Connected to boilermaker_brews.db: 6 tables. The %%sql command is ready.


**Test** — this should print `26`. If you get an error, read it bottom-up: `unable to open database file` almost always means the upload didn't happen or the filename differs.

In [3]:
%sql SELECT COUNT(*) AS n_products FROM products;

,n_products
0,26


**Load the self-checks** (provided — just run it):

In [4]:
# ---- Self-checks: run this cell once, then call check_qN(qN) after each exercise ----
# These give instant feedback. They are NOT the grade: Lab credit is for submitting;
# correctness is practiced here and *tested* on A1 and Practicum 1.

def _rows(r):
    assert r is not None, "Capture your result first:  %%sql q1 <<  (note the `q1 <<`)."
    return list(r)

def _cols(r):
    return [k.lower() for k in r.keys]

def check_q1(r):
    rows = _rows(r)
    assert len(rows) == 5, f"Expected 5 rows, got {len(rows)} - did you LIMIT 5?"
    assert "name" in _cols(r) and "price" in _cols(r), "Select exactly: name, price."
    print("[OK] Q1 - you just read a table on your own.")

def check_q2(r):
    rows = _rows(r)
    assert len(rows) == 9, f"Expected 9 espresso products, got {len(rows)} - check your WHERE."
    assert abs(rows[0][-1] - 5.75) < 1e-9, "First row should be the most expensive (5.75) - ORDER BY ... DESC."
    print("[OK] Q2 - filtering + sorting together.")

def check_q3(r):
    rows = _rows(r)
    assert len(rows) == 7, f"Expected 7 products, got {len(rows)} - BETWEEN is inclusive on both ends."
    print("[OK] Q3 - BETWEEN works.")

def check_q4(r):
    rows = _rows(r)
    assert len(rows) == 3, f"Expected 3 products, got {len(rows)} - use LIKE '%Tea%' (or '%tea%' - SQLite's LIKE is ASCII case-insensitive by default)."
    print("[OK] Q4 - pattern matching down.")

def check_q5(r):
    rows = _rows(r)
    assert len(rows) == 1, ("Expected one row holding one number, so use COUNT(*). "
                        "If you got no rows at all: '= NULL' matches nothing, "
                        "and there is a special verb for NULL.")
    v = rows[0][0]
    assert v != 0, "Got 0 - remember the lecture: `= NULL` matches nothing. Use IS NULL."
    assert v == 18203, f"Expected 18203 walk-ins, got {v}."
    print("[OK] Q5 - you dodged the NULL trap. 18,203 walk-ins found.")

def check_q6(r):
    v = _rows(r)[0][0]
    assert v == 15593, f"Expected 15593 app orders, got {v} - check your WHERE on channel."
    print("[OK] Q6 - exactly right: 15,593 app orders (~30%).")

def check_q7(r):
    rows = _rows(r)
    assert len(rows) == 5, f"Expected 5 rows, got {len(rows)}."
    assert str(rows[0][-1]).startswith("2026-08-31"), "First row should be the most recent order - ORDER BY order_ts DESC."
    print("[OK] Q7 - the freshest 5 orders.")

def check_q8(r):
    v = _rows(r)[0][0]
    assert v == 4822, f"Expected 4822, got {v} - join orders to stores and filter on the store *name*."
    print("[OK] Q8 (stretch) - your first JOIN! Discovery Park: 4,822 orders.")

print("Checks loaded. After each exercise, run its check cell.")


Checks loaded. After each exercise, run its check cell.


**Cheat sheet for this lab**

```sql
SELECT col1, col2      -- which columns
FROM   table_name      -- which table
WHERE  condition       -- which rows   (=, <, >, IN, BETWEEN, LIKE, IS NULL)
ORDER  BY col DESC     -- sort
LIMIT  5;              -- truncate
```

Tables: `stores(6)`, `products(26)`, `customers(600)`, `employees(36)`, `orders(~52k)`, `order_items(~79k)`. Full data dictionary on Brightspace.

### Q1. Peek at the menu

Show the **name and price** of the **first 5** products.

<details><summary>Hint</summary>

`SELECT ... FROM products LIMIT ...` — column names are `name`, `price`.

</details>

In [18]:
%%sql q1 <<
SELECT name, price
FROM products
LIMIT 5;
-- write your query here

,name,price
0,Drip Coffee 12oz,2.50
1,Drip Coffee 16oz,3.00
2,Cold Brew 16oz,4.25
3,Nitro Cold Brew,4.95
4,Espresso (double),3.25


In [19]:
check_q1(q1)

[OK] Q1 - you just read a table on your own.


### Q2. Espresso drinks, priciest first

List the **name and price** of every product in the `espresso` category, **most expensive first**.

<details><summary>Hint</summary>

Text values need quotes: `WHERE category = 'espresso'`. Sort with `ORDER BY ... DESC`.

</details>

In [21]:
%%sql q2 <<
SELECT name, price
FROM products
WHERE category = 'espresso'
ORDER BY price DESC

,name,price
0,Pumpkin Spice Latte,5.75
1,Peppermint Mocha,5.75
2,Caramel Latte,5.25
3,Mocha,5.25
4,Latte 16oz,5.00
5,Latte 12oz,4.50
6,Cappuccino,4.25
7,Americano,3.50
8,Espresso (double),3.25


In [22]:
check_q2(q2)

[OK] Q2 - filtering + sorting together.


### Q3. The middle of the menu

Which products cost **between \$3.00 and \$4.00 (inclusive)**? Show name and price.

<details><summary>Hint</summary>

`BETWEEN 3.00 AND 4.00` is inclusive on both ends.

</details>

In [25]:
%%sql q3 <<
SELECT name, price
FROM products
WHERE price>=3.00 AND price<=4.00
ORDER BY price DESC

,name,price
0,BB Sticker Pack,4.00
1,Americano,3.50
2,Butter Croissant,3.50
3,Espresso (double),3.25
4,Iced Black Tea,3.25
5,Blueberry Muffin,3.25
6,Drip Coffee 16oz,3.00


In [26]:
check_q3(q3)

[OK] Q3 - BETWEEN works.


### Q4. Tea time

Find every product whose **name contains the word `Tea`**.

<details><summary>Hint</summary>

`LIKE '%Tea%'` — `%` means "anything here". SQLite's default `LIKE` is case-insensitive for ASCII, so `'%tea%'` finds the same rows - try it. (Don't assume every database behaves this way; some are case-sensitive by default.)

</details>

In [27]:
%%sql q4 <<
SELECT name
FROM products
WHERE name LIKE '%Tea%'
ORDER BY price DESC


,name
0,Iced Black Tea
1,Earl Grey Tea
2,Green Tea


In [28]:
check_q4(q4)

[OK] Q4 - pattern matching down.


#### Try it (no checker): `DISTINCT`

What **categories** exist on the menu? `SELECT DISTINCT category FROM products;` — expect 6. Also try `DISTINCT loyalty_tier` on `customers` (expect 3). First move on any unfamiliar column!

In [30]:
%%sql
SELECT DISTINCT loyalty_tier FROM customers

,loyalty_tier
0,silver
1,none
2,gold


### Q5. Count the walk-ins

How many orders were placed by **walk-in customers** (no loyalty account on the order)? Return a single count.

*This is the trap from lecture. If your first try returns 0 — you've found it.*

<details><summary>Hint</summary>

Walk-ins have `customer_id` **NULL**, and `= NULL` never matches. There's a special verb for this.

</details>

In [33]:
%%sql q5 <<
SELECT count(*)
FROM orders
WHERE customer_id is NULL



,count(*)
0,18203


In [34]:
check_q5(q5)

[OK] Q5 - you dodged the NULL trap. 18,203 walk-ins found.


### Q6. App adoption

How many orders came through the **app** channel?

<details><summary>Hint</summary>

The `orders.channel` column holds `'counter'` or `'app'`.

</details>

In [35]:
%%sql q6 <<
SELECT count(*)
FROM orders
WHERE channel = 'app'


,count(*)
0,15593


In [36]:
check_q6(q6)

[OK] Q6 - exactly right: 15,593 app orders (~30%).


### Q7. The freshest five

Show the **order_id and order_ts of the 5 most recent orders**.

<details><summary>Hint</summary>

Sort by the timestamp column, newest first, then truncate.

</details>

In [46]:
%%sql q7 <<
SELECT order_id, order_ts
FROM orders
ORDER BY order_ts DESC
LIMIT 5;

,order_id,order_ts
0,51779,2026-08-31 19:48:35
1,51767,2026-08-31 19:38:11
2,51821,2026-08-31 19:37:18
3,51782,2026-08-31 19:33:25
4,51920,2026-08-31 19:10:42


In [47]:
check_q7(q7)

[OK] Q7 - the freshest 5 orders.


### Q8. STRETCH — your first JOIN

How many orders were placed at the **Discovery Park** store? Return a single count.

*Stretch: your first JOIN, from SQL Lec 3 (last week's lectures).*

<details><summary>Hint</summary>

```sql
FROM orders o
JOIN stores s ON o.store_id = s.store_id
``` then filter on `s.name`.

</details>

In [51]:
%%sql q8 <<
SELECT count(*)
FROM orders o
JOIN stores s ON o.store_id = s.store_id
WHERE s.name = 'Discovery Park'


,count(*)
0,4822


In [52]:
check_q8(q8)

[OK] Q8 (stretch) - your first JOIN! Discovery Park: 4,822 orders.


## Important details and common questions

- **Rows have no guaranteed order without `ORDER BY`.** A result that happens to look sorted is not a contract.
- **`LIKE` depends on the database.** In this SQLite lab, ASCII letters are case-insensitive by default; Unicode and other database systems may behave differently.
- **`NULL` is not a value.** Use `IS NULL` or `IS NOT NULL`.
- **Boolean precedence:** `NOT` is evaluated before `AND`, and `AND` before `OR`. Use parentheses whenever mixed logic could be read two ways.
